# PV Tilt

Compare eddy tilt direction with planetary, topographic, and total shallow-water PV-gradient directions.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import seacofs_tilt_tools as tilt

paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
df_eddies, df_tilt = tilt.load_tilt_tables(paths, add_regions=True, grid=grid)
df_eddies = tilt.add_pv_gradient_terms(df_eddies, grid)

df_eddies[["Eddy", "Day", "Cyc", "Region", "TiltDis", "TiltDir", "PV_grad_mag", "dtheta_PV_grad"]].head()


In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(13, 3.6), sharey=True, constrained_layout=True)

panels = [
    ("beta", r"$\beta=f_y$ (s$^{-1}$ m$^{-1}$)", None, True),
    ("PV_grad_mag", r"$|\nabla PV|$", (0, 3e-13), False),
    ("PV_grad_topo_mag", r"$|\nabla PV|_{topo}$", (0, 3e-13), False),
    ("PV_grad_plan_mag", r"$|\nabla PV|_{plan}$", None, False),
]

for ax, (col, label, xlim, linfit) in zip(axs, panels):
    tilt.binned_tilt_panel(ax, df_eddies, col, label, xlim=xlim, linfit=linfit)
axs[0].set_ylabel("Tilt distance (km)")
axs[0].legend(frameon=False)
plt.show()


In [ ]:
theta_cols = ["dtheta_PV_grad", "dtheta_PV_grad_topo", "dtheta_PV_grad_plan"]
row_labels = ["Total PV gradient", "Topographic PV gradient", "Planetary PV gradient"]
region_groups = [["S1", "S2"], ["U1", "U2"], ["D1", "D2"]]
region_titles = ["Shelf", "Upstream", "Downstream"]
bins = np.arange(0, 181, 10)

fig, axs = plt.subplots(3, 3, figsize=(12, 8.5), sharex=True, constrained_layout=True)
for r, (theta_col, row_label) in enumerate(zip(theta_cols, row_labels)):
    for c, regs in enumerate(region_groups):
        ax = axs[r, c]
        for cyc, color in [("AE", "darkred"), ("CE", "navy")]:
            vals = df_eddies.loc[(df_eddies.Cyc == cyc) & (df_eddies.Region.isin(regs)), theta_col].dropna()
            ax.hist(vals, bins=bins, histtype="stepfilled", alpha=0.22, color=color)
            ax.hist(vals, bins=bins, histtype="step", linewidth=1.8, color=color, label=cyc if (r, c) == (0, 0) else None)
        ax.axvline(90, color="0.5", linestyle=":")
        ax.set_xlim(0, 180)
        if r == 0:
            ax.set_title(region_titles[c])
        if c == 0:
            ax.set_ylabel(row_label)

for ax in axs[-1]:
    ax.set_xticks([0, 90, 180])
    ax.set_xticklabels(["Aligned", "Perpendicular", "Opposite"])
axs[0, 0].legend(frameon=False)
plt.show()


In [ ]:
df_plot = df_eddies.loc[df_eddies.TiltDis > 20].copy()
row_filters = [
    ("All eddies", np.ones(len(df_plot), dtype=bool)),
    ("Topographic dominated", df_plot.topo_plan_ratio > 0),
    ("Planetary dominated", df_plot.topo_plan_ratio < 0),
]

fig, axs = plt.subplots(3, 3, figsize=(12, 8.5), sharex=True, constrained_layout=True)
for r, (row_label, mask) in enumerate(row_filters):
    row = df_plot.loc[mask]
    for c, regs in enumerate(region_groups):
        ax = axs[r, c]
        for cyc, color in [("AE", "darkred"), ("CE", "navy")]:
            vals = row.loc[(row.Cyc == cyc) & (row.Region.isin(regs)), "dtheta_PV_grad"].dropna()
            ax.hist(vals, bins=bins, histtype="stepfilled", alpha=0.22, color=color)
            ax.hist(vals, bins=bins, histtype="step", linewidth=1.8, color=color, label=cyc if (r, c) == (0, 0) else None)
        ax.axvline(90, color="0.5", linestyle=":")
        ax.set_xlim(0, 180)
        if r == 0:
            ax.set_title(region_titles[c])
        if c == 0:
            ax.set_ylabel(row_label)
for ax in axs[-1]:
    ax.set_xticks([0, 90, 180])
    ax.set_xticklabels(["Aligned", "Perpendicular", "Opposite"])
axs[0, 0].legend(frameon=False)
plt.show()


## PV-gradient regional rose plot

The same regional rose layout can be used with PV-gradient magnitude and direction.


In [ ]:
pv_mag_bins = [0, 1e-14, 5e-14, 1e-13, 3e-13, np.inf]
fig, rose = tilt.rose_plot(
    df_eddies.dropna(subset=["PV_grad_mag", "PV_grad_theta"]),
    grid,
    mag="PV_grad_mag",
    theta="PV_grad_theta",
    mag_bins=pv_mag_bins,
    frac=2.6,
    direction_offset=0,
    shelf_offset=80,
    legend_title=r"$|\nabla PV|$",
)
